Task : Write a script (using PySpark/Spark SQL) to rank employees within each department by their Monthly_Sales in highest sales first.

In [0]:
sales_df = spark.read.format("csv")\
    .option("Header","true")\
    .option("inferSchema","true")\
    .load("/Volumes/sql_problems/default/my_volume/day06_EmployeeSales.csv")

display(sales_df)

In [0]:
sales_df.createOrReplaceTempView("sales")

In [0]:
%sql
SELECT 
    Department,
    name,
    sales,
    DENSE_RANK(sales) OVER(
    PARTITION BY Department
    ORDER BY sales DESC
) as rankings
FROM sales
ORDER BY Department, rankings

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank

windowSpec = Window.partitionBy(col("Department"))\
    .orderBy(col("sales").desc())

# Step 2: Use Dense_Rank() and assign to a new DataFrame variable
result_df = sales_df.withColumn("rankings", dense_rank().over(windowSpec))

result_df.select(
    "name",
    "Department",
    "sales",
    "rankings"
)
display(result_df)